# Generazione Dataset avanzato con architettura Multi-Agente (Actor-Critic)

# Descrizione del problema

All'interno di questo Jupyter Notebook è presente una versione alternativa del Jupyter 02.

La soluzione adottata nello scorso Jupyter è stata utilizzare Gemma4, una famiglia di modelli linguistici di grandi dimensioni (Large Language Model) sviluppata da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di non limitarmi nella scrittura di un prompt per la generazione di n ticket, ma strutturare una funzione che, passati parametri come categoria, priorità, colloquialità, ecc... creasse in **run time** dei prompt casuali per la generazione di **singoli ticket**, per simulare le richieste dei singoli utenti, nella vita reale tutti diversi a modo loro in quanto esseri umani.

Questa nuova versione presenta un architettura Multi-Agente chiamata in letteratura Actor-Critic. Invece di sperare che l'LLM faccia tutto giusto al primo colpo, si creano dei sistemi in cui i modelli "ragionano" e si correggono a vicenda.

Ecco come sarà divisa l'architettura:

- Agente 1 (**Il Generatore**): Riceve i parametri casuali (categoria, priorità, ecc.) e ha il compito di generare il ticket. Questo corrisponde per intero all'architettura analizzata nel Jupyter 02.

- Agente 2 (**Il Revisore**): Riceve in input il testo generato dall'Agente 1 e un prompt di validazione molto rigido. Ad esempio: "Agisci come un revisore qualità. Leggi il seguente ticket e miglioralo in modo da sembrare più realistico".

- **Un Ciclo di Feedback**: Se l'Agente 2 restituisce true, il ticket viene salvato nel CSV e si passa al successivo. Se restituisce false, il ciclo riparte e l'Agente 1 ritenta la generazione (magari passandogli come feedback il "motivo" dell'errore, con un limite massimo di 3 tentativi per evitare loop infiniti).

### Come eseguire il codice di questo Jupyter Notebook

__Questo Jupyter Notebook contiene già del codice eseguito a scopo dimostrativo, tuttavia a seguire è disponibile una breve guida passo-passo qualora si desiderasse configurare l'ambiente ed eseguirlo nuovamente__

*Questa sezione è effettivamente la più complessa da replicare autonomamente, in quanto è richiesta l'installazione di un'applicazione e di un modello di mediamente una decina di GB. Potrebbe inoltre essere richiesta una maggiore potenza di calcolo. Nonostante questo, esso è comunque perfettamente replicabile per intero.*

Per eseguire correttamente questo Notebook e avviare la generazione del dataset tramite modello LLM in locale, occorre seguire questi passaggi preliminari:
- Ollama deve essere installato direttamente sul sistema operativo per poter ospitare il modello in locale. Qualsiasi sia il sistema operativo, basta scaricare e installare l'applicazione ufficiale dal sito [ollama.com](https://ollama.com).
- Ad installazione finita, dovrà essere scaricato ed installato il modello. Da powershell, basta runnare il comando: *ollama run gemma4:e2b* (7.2 GB richiesti)
- A fine procedura, qualora si desiderasse rimuovere il modello, basterà utilizzare il comando: *ollama rm gemma4:e2b*

_Installazione di librerie utilizzate nel progetto:_ 

In [120]:
%pip install pandas ollama pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Definizione degli agenti

Import delle librerie:

In [121]:
import json
import pandas as pd
import random
from ollama import Client
from pydantic import BaseModel, Field

## Agente 1: Il generatore

Questa sarà la struttura dell'Agente 1, ovvero il responsabile della generazione dei ticket.

In [122]:
class TicketSintetico(BaseModel):
    title: str = Field(
        description="Un titolo breve, specifico e realistico."
    )
    body: str = Field(
        description=(
            "Il corpo del ticket. Deve contenere una descrizione breve (1-2 frasi) del problema reale. "
            "NON usare placeholder, parentesi quadre, simboli di markup o istruzioni di compilazione (es. '[Descrivere qui...]'). "
            "Non è obbligatorio inserire nominativi di applicazioni/servizi ma, se lo fai, utilizza nomi verosimili (non usare ad esempio modulo X o endpoint x)"
            "Il testo deve essere testo pronto, discorsivo e già compilato, scritto dal punto di vista dell'utente o del dipendente che sta riscontrando il problema in prima persona"
        )
    )

Funzione della generazione di un singolo ticket. Proprio come spiegato nel dettaglio nel Jupyter 2, qui viene formattato il prompt della richiesta da fare al modello per la creazione di un determinato ticket basandoci su dei parametri.

L'unica modifica effettuata è l'aggiunta di un parametro "genere" che risolve il problema del gender bias dei LLM, dove viene sempre dato per scontato che chi parli sia di sesso maschile. Ovviamente, la modifica verrà inserita anche all'interno del ciclo che genererà i ticket.

In [123]:
client = Client()
def genera_ticket_con_gemma(categoria, priorita,esperienza,colloquialità,incipit,titolo,genere="",disturbo=False,typo=False):
    prompt_utente = f"""Genera un ticket di assistenza aziendale in lingua italiana con queste caratteristiche:
    - Categoria del ticket: {categoria}. Questo rappresenta l'argomento principale di cui deve parlare il ticket.
    - Priorità del problema: {priorita}.
    - Livello di esperienza dell'utente: {esperienza}.
    - Livello di colloquialità/tono: {colloquialità}
    - Genere del mittente: Chi scrive è {genere}. Declina OBBLIGATORIAMENTE tutti gli aggettivi e i verbi riferiti a te stesso al {genere}
    STILE DI SCRITTURA:
    1. Sii estremamente sintetico. Il testo deve essere composto da massimo 2-3 frasi corte.
    2. Scrivi in prima persona. 
    3. Non usare MAI caratteri speciali di formattazione come asterischi (*), corsivi o grassetti.
    4. Evita deliri o flussi di coscienza lunghi: vai dritto al problema reale.
    5. La forma di saluto iniziale deve essere esattamente "{incipit}" (se non è vuota) o una sua variante naturale.
    6. Usa termini tecnici, acronimi o parole chiave specifiche in base alla categoria del ticket ({categoria}) e all'esperienza ({esperienza}).
    7. Il titolo deve simulare l'oggetto scritto da un utente reale (breve, sbrigativo o imperfetto). Inizia il titolo UTILIZZANDO OBBLIGATORIAMENTE questa esatta espressione iniziale: "{titolo}". Continua poi la frase completando l'oggetto in modo che si colleghi perfettamente alla categoria ({categoria}) e soprattutto alla priorità ({priorita}).
    8. Rispetta rigorosamente la coerenza: se la priorità è Bassa, non inserire scadenze imminenti o blocchi operativi nel titolo o nel corpo
    """
    if disturbo:
        prompt_utente+="""\n - VINCOLO DI CONTESTO DINAMICO (OBBLIGATORIO):
                        - {disturbo}. Integra questo dettaglio nel testo in modo naturale, usandolo come elemento di sfondo, senza farlo diventare il problema principale."""
    if typo:
        prompt_utente += "\n- Inserisci dove rendi opportuno un typo per rendere il ticket più human like."
    try:
        response = client.chat(
            model='gemma4:e2b', 
            messages=[{'role': 'user', 'content': prompt_utente}],
            format=TicketSintetico.model_json_schema(),
            options={
                'keep_alive': -1,  # -1 dice a Ollama: "Non scaricare MAI il modello finché non si chiude lo script"
                'temperature': 0.4
            }
        )
        dati_ticket = json.loads(response['message']['content'].strip())
        return dati_ticket['title'], dati_ticket['body']
    except Exception as e:
        print(f"Errore durante la generazione locale: {e}")
        return None, None


## Agente 2: Il revisore

Struttura dell'output richiesto dal revisore. Viene aggiunto il campo *modifiche_applicate*, che non sarà altro che un booleano che indica se il revisore ha fatto o meno delle modifiche. Si potrebbe potenzialmente rimuovere il campo **modifiche_effettuate** in cui si chiede di descrivere cosa ha effettivamente modificato.

In [124]:
class TicketRevisionato(BaseModel):
    titolo: str = Field(
        description=(
            "Il titolo del ticket corretto e ripulito. "
            "Deve essere privo di markdown, formattazioni speciali o placeholder."
        )
    )
    corpo: str = Field(
        description=(
            "Il corpo del ticket corretto e ripulito dai soli artefatti dell'IA. "
            "DEVE conservare rigorosamente lo stesso tono emotivo, l'informalità, i deliri o l'ansia dell'originale. "
            "Non renderlo professionale o freddo se in origine non lo era."
        )
    )
    modifiche_effettuate: str = Field(
        description="Descrivi in una frase breve quale tipo di rumore o imperfezione umana hai aggiunto per rendere il testo meno robotico."
    )

Funzione della revisione di un singolo ticket.

In [125]:
def revisione_ticket_con_gemma(titolo_grezzo, corpo_grezzo):
    prompt_critico = f"""Agisci come un Data Augmentation Specialist per un dataset di Machine Learning.
    Il tuo compito è prendere il seguente ticket grezzo e modificarlo attivamente per renderlo meno artificiale possibile.

    TICKET GREZZO:
    Titolo: {titolo_grezzo}
    Corpo: {corpo_grezzo}

    DIRETTIVE OPERATIVE (TASSATIVE):
    1. Elimina qualsiasi carattere markdown o formula da IA (es. "Ecco il ticket").
    2. **INIETTA RUMORE E IMPERFEZIONI**: Se il testo sembra scritto da un assistente virtuale o è troppo corretto/formale, devi sporcarlo! Aggiungi un tono più umano. comunque mantieni la frase corretta grammaticalmente parlando (non inserire imperfezioni grammaticali)
    3. Il dataset di Machine Learning ha bisogno di variabilità e imperfezioni linguistiche per evitare l'overfitting: non avere paura di rendere il testo meno pulito.
    4. VARIETÀ LESSICALE: Quando inietti rumore (esitazioni, fretta, rabbia), NON usare sempre le parole 'casino', 'proprio', 'davvero' o 'grosso problema'. Usa sinonimi, cambia struttura, usa frasi tronche, esclamazioni diverse o semplicemente cattiva punteggiatura.
    5. se all'interno del ticket o del titolo vedi le parole "priorità alta, media o bassa", sostituiscile con qualcos'altro di inerente
    Restituisci il ticket modificato e arricchito in formato JSON.
    """

    try:
        response = client.chat(
            model='gemma4:e2b', 
            messages=[{'role': 'user', 'content': prompt_critico}],
            format=TicketRevisionato.model_json_schema(),
            options={
                'keep_alive': -1,
                'temperature': 0.1
            }
        )
        dati_validazione = json.loads(response['message']['content'].strip())
        
        return dati_validazione['titolo'], dati_validazione['corpo'], dati_validazione['modifiche_effettuate']
        
    except Exception as e:
        print(f"Errore durante la validazione locale: {e}")
        return titolo_grezzo, corpo_grezzo, "-"

# Sviluppo dell'architettura

In [126]:
categorie_disponibili = ["Tecnico", "Amministrazione", "Commerciale"]
priorita_disponibili = ["Bassa", "Media", "Alta"]
livelli_esperienza = [
    "Principiante", "Intermedio", "Esperto/Professionale", "Utente occasionale",
    "Primo utente", "Avanzato", "Primo utilizzo", "Utente che si crede esperto ma non lo è",
    "Tecnofobo / Anziano", "Nativo digitale", "Sviluppatore / Tecnico interno",     
    "Amministratore di sistema (Admin)","Utente pigro","Cliente VIP / Direzione",            
    "Utente metodico/Preciso","Ex-utente di un software concorrente"
]
livelli_colloquialita = [
    "Formale e distaccato", "Molto informale e colloquiale", "Frustrato/Arrabbiato",
    "Confuso", "Dettagliato", "Frettoloso", "Sarcastico", "Simpaticone", "Ansioso / Nel panico",
    "Disperato / Rassegnato", "Logorroico / Prolisso", "Telegrafico / Minimale","Passivo-aggressivo",
    "Gentile ed empatico", "Disorganizzato / Sconnesso", "Diffidente / Sospettoso"
]
pool_incipit_urgenti = [
    "Così non posso lavorare! ", 
    "Non so più che fare... ", 
    "Da stamattina sono completamente bloccato a ",
    "Ho un'urgenza assoluta: ",
    "Scrivo perché l'attività è interrotta: ",
    "Ho bisogno di risolvere subito ",
    "Siamo completamente fermi su ",
    "Blocco totale della produzione su ",
    "Disastro, non funziona più nulla dopo ",
    "Soluzione immediata richiesta per ",
    "È prioritario sbloccare ",
    "Impossibile procedere con il lavoro a causa di ",
    "Mi serve supporto ora, sono fermo su ",
    "Situazione critica su ",
    "Non si può andare avanti così! "
]

pool_incipit_normali = [
    "",
    "Ciao, ", 
    "Buongiorno. ", 
    "Buongiorno! ",
    "Eccomi di nuovo, ", 
    "Salve, ", 
    "Buongiorno, ", 
    "Scusate il disturbo, ", 
    "Ho un problema: ", 
    "Vi scrivo perché ", 
    "Creo questo ticket per ", 
    "Da stamattina sto provando a ", 
    "Dopo l'ultimo aggiornamento, ", 
    "Ho già provato a riavviare ", 
    "Potete dare un occhiata a ", 
    "Ho già provato a riavviare, ma ", 
    "Gentile supporto, vi contatto per ", 
    "Vorrei segnalare ", 
    "Ho necessità di segnalare ",
    "Avrei bisogno di un controllo su ",
    "Vi contatto per una anomalia riscontrata su ",
    "Riscontro un errore nel sistema quando provo a ",
    "Non riesco a completare la procedura di ",
    "Sembra esserci un bug su ",
    "Eccoci di nuovo... "
]

pool_incipit_bassa = [
    "",
    "Quando avete tempo, ",
    "Senza alcuna fretta, ",
    "Giusto per informazione, ",
    "Vi scrivo per una questione non urgente: ",
    "Avrei una curiosità su ",
    "Segnalo una piccolissima anomalia: ",
    "Un piccolo dubbio su ",
    "Nulla di bloccante, ma volevo segnalare che ",
    "Per il futuro sarebbe utile capire se ",
    "Solo una domanda di routine: ",
    "Quando capita, potreste dare un'occhiata a ",
    "Solo una richiesta di info su ",
    "Piccolo chiarimento per quando siete liberi: ",
    "Volevo solo lasciarvi un feedback riguardo a "
]

pool_titolo_urgenti = [
    "",
    "Urgente: non va ", 
    "Richiesta urgente: ", 
    "Assistenza immediata per ",
    "BLOCCO TOTALE: ",
    "CRITICO - Malfunzionamento su ",
    "EMERGENZA: ",
    "Interruzione servizio su "
]

pool_titolo_normali = [
    "",
    "Problema con ", 
    "Aiuto: ", 
    "Errore ", 
    "Bloccato su ", 
    "Info per ", 
    "Non va ", 
    "Chiarimento su ", 
    "Mancato funzionamento ", 
    "Procedura per ", 
    "Richiesta chiarimenti ",
    "Segnalazione errore su ",
    "Anomalia riscontrata in ",
    "Richiesta di supporto per "
]

pool_titolo_bassa = [
    "",
    "Richiesta info (non urgente): ",
    "Segnalazione minore su ",
    "Domanda generica: ",
    "Feedback su ",
    "Piccolo dubbio su ",
    "Suggerimento / Miglioria per ",
    "Info di routine per ",
    "Nota di aggiornamento su ",
    "Curiosità tecnica riguardo a "
]
reparti_aziendali = [
    "Reparto Logistica", "Ufficio Risorse Umane", "Ufficio Acquisti", 
    "Produzione", "Marketing", "Sede di Roma", "Filiale Estera", 
    "Reparto tecnico", "Cliente",
    "Ufficio Contabilità", "Controllo Qualità", "Ufficio Legale", 
    "Reparto Vendite", "Sede di Milano", "Assistenza Clienti", 
    "Ufficio Personale", "Management", "Direzione Generale", 
    "Fornitore Esterno"
]

strumenti_generici = [
    "gestionale interno", "portale dipendenti", "piattaforma cloud", 
    "software di rete", "sistema centrale", "CRM", "applicazione", 
    "servizio cloud",
    "database aziendale", "server centrale", "portale fornitori", 
    "applicazione di fatturazione", "pannello di controllo", 
    "sistema di login", "software di contabilità", "piattaforma HR", 
    "modulo richieste", "archivio digitale"
]

Il seguente ciclo è identico a quello analizzato nello scorso Jupyter, ma in più viene aggiunta la fase di revisione dell'Agente 2. Inoltre, è presente un if che indica se il revisore ha effettuato modifiche o meno.

In [127]:
import pandas as pd
import random

percorso_csv = "dataset/ticket_actor_critic.csv"

da_fare = 95

#controllo se è il primo inserimento o meno
try:
    df_esistente = pd.read_csv(percorso_csv)
    
    if not df_esistente.empty and 'id' in df_esistente.columns:
        ultimo_id_str = df_esistente['id'].iloc[-1]
        id_di_partenza = int(ultimo_id_str.split("-")[1]) + 1
    else:
        id_di_partenza = 0
        
    scrivi_header = False
    print(f"File CSV rilevato con Pandas. Riprendo dall'ID: TK-{id_di_partenza}")

except (FileNotFoundError, pd.errors.EmptyDataError):
    #parto da zero se il file non è stato trovato o è vuoto
    id_di_partenza = 0
    scrivi_header = True
    print("File CSV non trovato o vuoto. Creazione nuovo dataset da TK-0.")

print("-" * 50)

for i in range(0, da_fare):
    id_corrente_numerico = id_di_partenza + i
    stringa_id = f"TK-{id_corrente_numerico}"
    
    categoria = random.choice(categorie_disponibili)
    priorità = random.choice(priorita_disponibili)
    esperienza = random.choice(livelli_esperienza)
    colloquialità = random.choice(livelli_colloquialita)
    attiva_typo = random.choices([True, False], weights=[20, 80])[0]
    genere = genere_utente = random.choice(["Maschio", "Femmina"])
    #si sceglie incipit in base alla priorità. si usa la meccanica dei pesi per creare rumore
    if priorità == "Alta":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali], weights=[70, 30])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali], weights=[70, 30])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Media":
        pool_scelta_titolo = random.choices([pool_titolo_urgenti, pool_titolo_normali, pool_titolo_bassa], weights=[20, 60, 20])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_urgenti, pool_incipit_normali, pool_incipit_bassa], weights=[20, 60, 20])[0]
        incipit = random.choice(pool_scelta_incipit)

    elif priorità == "Bassa":
        pool_scelta_titolo = random.choices([pool_titolo_normali, pool_titolo_bassa], weights=[30, 70])[0]
        titolo = random.choice(pool_scelta_titolo)
        
        pool_scelta_incipit = random.choices([pool_incipit_normali, pool_incipit_bassa], weights=[30, 70])[0]
        incipit = random.choice(pool_scelta_incipit)    

    disturbo = ""
    if random.randint(0,2)==1:
        if categoria == "Tecnico":
            disturbo = f"L'utente specifica di lavorare presso: {random.choice(reparti_aziendali)}"
        elif categoria == "Amministrazione":
            disturbo = f"L'operazione contabile avviene tramite lo strumento: {random.choice(strumenti_generici)}"
        else:  # Commerciale
            disturbo = f"L'utente menziona che l'offerta serve per il budget del: {random.choice(reparti_aziendali)} o per lo strumento  {random.choice(strumenti_generici)}"

    print(f"Generazione in corso... [{categoria} - {priorità} - {esperienza} - {colloquialità} - {attiva_typo}] ({i+1}/{da_fare})")
    titolo_bozza, corpo_bozza = genera_ticket_con_gemma(categoria, priorità, esperienza, colloquialità,incipit,titolo,genere,disturbo, attiva_typo)
    if titolo_bozza and corpo_bozza:
        print("Bozza creata. Avvio revisione")
        titolo_finale, corpo_finale ,modifiche_effettuate = revisione_ticket_con_gemma(titolo_bozza,corpo_bozza)
        singolo_ticket = {
            "id": [stringa_id],
            "title": [titolo_finale],
            "body": [corpo_finale],
            "category": [categoria],
            "priority": [priorità],
            "modifiche_effettuate": [modifiche_effettuate]
        }
        df_singolo = pd.DataFrame(singolo_ticket)
        
        #se è il primo ticket assoluto creato da zero, scrive l'header e apre in 'w'
        if id_corrente_numerico == 0 and scrivi_header:
            df_singolo.to_csv(percorso_csv, index=False, mode='w', header=True)
            scrivi_header = False 
        else:
            df_singolo.to_csv(percorso_csv, index=False, mode='a', header=False)

        print(f"Ticket {stringa_id} Creato e SALVATO nel CSV!")
        print(f"Titolo: {titolo_finale}")
        print(f"Corpo:  {corpo_finale[:80]}...") 
        print("-" * 50)
        print()
    else:
        print("Generazione fallita. Salto al successivo.")
        print("-" * 50)

print(f"\nGenerazione conclusa con successo!")
print(f"Trovi tutti i dati aggiornati in tempo reale qui: '{percorso_csv}'.")


File CSV rilevato con Pandas. Riprendo dall'ID: TK-407
--------------------------------------------------
Generazione in corso... [Amministrazione - Media - Nativo digitale - Sarcastico - False] (1/95)
Bozza creata. Avvio revisione
Ticket TK-407 Creato e SALVATO nel CSV!
Titolo: CRITICO! Problemi con le fatture
Corpo:  Ciao supporto, vi scrivo perché c'è un grosso malfunzionamento nella gestione de...
--------------------------------------------------

Generazione in corso... [Amministrazione - Alta - Primo utente - Sarcastico - False] (2/95)
Bozza creata. Avvio revisione
Ticket TK-408 Creato e SALVATO nel CSV!
Titolo: Urgente: la gestione documenti amministrazione non funziona
Corpo:  Ho un'urgenza, la piattaforma di amministrazione non funziona per me. Sono una p...
--------------------------------------------------

Generazione in corso... [Commerciale - Bassa - Intermedio - Sarcastico - False] (3/95)
Bozza creata. Avvio revisione
Ticket TK-409 Creato e SALVATO nel CSV!
Titolo: Dobb

# Conclusioni

L'esperimento condotto in questo notebook ha dimostrato come ottimizzare la generazione di testi tramite LLM. In particolare, la soluzione analizzata è stata quella di sviluppare un'architettura Multi-Agente per applicare rigorose tecniche di Data Curation.


Il componente principale di questa soluzione è sicuramente "il revisore", il quale non si limita a correggere gli artefatti tecnici dell'IA (come formattazione markdown o formule di cortesia robotiche). È stato configurato in modo proattivo per iniettare "rumore" umano: frustrazione, urgenza, intercalari. Questa entropia linguistica è essenziale per prevenire l'overfitting dei futuri classificatori NLP.

Questa procedura ovviamente va a dimezzare la velocità di creazione dei ticket, in quanto l'architettura raddoppi le chiamate ad ollama.

Diamo un'occhiata all'output finale, con lo scopo di validarne la qualità. All'interno della pipeline di Machine Learning sarà necessario svolgere un'EDA avanzata dei dati.

In [128]:
import pandas as pd
df_anteprima = pd.read_csv("dataset/ticket_actor_critic.csv")
print("*** TICKET ACTOR CRITIC ***")
df_anteprima.sample(10)

*** TICKET ACTOR CRITIC ***


,id,title,body,category,priority,modifiche_effettuate
65,TK-67,Domanda sulla configurazione iniziale della fa...,"Ciao, avrei una domanda sulla configurazione d...",Commerciale,Bassa,Il testo è stato modificato per introdurre inf...
323,TK-325,Ho problemi ad accedere al modulo amministrativo,"Buongiorno, ho difficoltà proprio ad entrare n...",Amministrazione,Bassa,Aggiunti elementi di esitazione e frustrazione...
138,TK-140,"URGENTE: Problemi di deploy su staging, non ri...",Non riesco proprio a completare il deploy del ...,Tecnico,Media,Il testo è stato modificato per introdurre un ...
265,TK-267,URGENTE: Problemi con licenze e i costi commer...,Ho bisogno di una soluzione subito per gestire...,Commerciale,Alta,Il testo è stato modificato per introdurre un ...
324,TK-326,Blocco totale della produzione su licenze soft...,Il sistema di pricing commerciale ha un errore...,Commerciale,Alta,Aggiunti elementi di urgenza e frustrazione (e...
131,TK-133,"Problemi con l'accesso ai dati contabili, è ur...",Sono completamente bloccato da stamattina a ca...,Amministrazione,Alta,Il testo è stato modificato per introdurre un ...
99,TK-101,Problemi configurazione amministrazione,Sono completamente bloccato da stamattina a ce...,Amministrazione,Media,Il testo è stato modificato per introdurre imp...
129,TK-131,Problema con il setup del report mensile,"Ciao, ho notato un'anomalia nel setup del repo...",Amministrazione,Media,Il testo è stato modificato per introdurre un ...
446,TK-448,"Problemi con le fatture medie, aiuto!",Ho un casino con l'inserimento dei dati contab...,Amministrazione,Media,"Aggiunti tono urgente, linguaggio più colloqui..."
63,TK-65,CRITICO: Problemi sul server di produzione,"Dopo l'ultimo aggiornamento, ho riscontrato un...",Tecnico,Alta,"Aggiunti tono informale, espressioni più collo..."


## Possibili Miglioramenti

Questo script mostra diversi possibili miglioramenti:
- Diversa strutturazione dei prompt: anche qui è possibile utilizzare diversi approcci di prompt engineering per avere risultati diversi e adatti alla situazione

__Una EDA più accurata è disponibile all'interno del Jupyter 03.__